# Color/Shape Final Data Preparation

Run the final Color/Shape loading, accuracy, and planned correct-trial RT-cleaning pipeline.

In [ ]:
from pathlib import Path
import importlib
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = REPO_ROOT / "code" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import posthoc_analysis.color_shape as color_shape
color_shape = importlib.reload(color_shape)

print("Repository root:", REPO_ROOT)
print("Source directory:", SRC_DIR)

## Load, Annotate, and Compute Accuracy

Load the retained Color/Shape subjects, keep main-task rows only, add congruency and value-difference columns, and compute subject/session congruent and incongruent accuracy.

In [ ]:
color_shape_results = color_shape.load_annotate_and_compute_color_shape_accuracy(
    save_outputs=True
)

color_shape_data = color_shape_results["data"]
color_shape_annotated = color_shape_results["annotated_data"]
color_shape_accuracy = color_shape_results["accuracy_summary"]
color_shape_loading_log = color_shape_results["loading_log"]
color_shape_mapping_log = color_shape_results["mapping_log"]

print("Loaded main-task rows:", color_shape_data.shape)
print("Annotated main-task rows:", color_shape_annotated.shape)
print("Accuracy summary:", color_shape_accuracy.shape)

## Loading Checks

Inspect retained/excluded subjects and final trial counts.

In [ ]:
display(color_shape_loading_log["status"].value_counts(dropna=False).rename("n"))

display(
    color_shape_mapping_log.loc[
        color_shape_mapping_log["mapping_status"] == "excluded",
        ["subject_id", "group", "run1", "run2", "mapping_notes"],
    ]
)

display(
    color_shape_data.groupby(["group", "session"], dropna=False)
    .size()
    .rename("n_rows")
    .reset_index()
)

display(
    color_shape_annotated.groupby(["group", "session", "congruency"], dropna=False)
    .size()
    .rename("n_rows")
    .reset_index()
)

display(color_shape_annotated[["relevant_val_diff", "irrelevant_val_diff"]].describe())

## Accuracy Summary

Accuracy uses all retained main-task trials and is summarized at the subject/session level.

In [ ]:
display(color_shape_accuracy)

display(
    color_shape_accuracy.groupby(["group", "session"], dropna=False)[
        ["acc_congruent", "acc_incongruent"]
    ]
    .agg(["mean", "std", "count"])
)

## Planned Correct-Trial RT Pipeline and Analysis Log

For RT analyses, keep correct response trials only, remove RT values below 150 ms, remove within-subject/session RT outliers using 3 MAD, compute congruent/incongruent RT, and create the subject/session analysis log.

In [ ]:
rt_results = color_shape.load_compute_color_shape_rt_and_analysis_log(
    save_outputs=True,
    min_rt_ms=150,
    mad_threshold=3,
    bin_width_ms=150,
)

color_shape_rt_planned_cleaned = rt_results["planned_rt_cleaned_data"]
color_shape_rt_low_rt_summary = rt_results["planned_rt_low_rt_summary"]
color_shape_rt_mad_summary = rt_results["planned_rt_mad_summary"]
color_shape_rt_overall_removal_summary = rt_results["planned_rt_overall_removal_summary"]
color_shape_rt_summary = rt_results["reaction_time_summary"]
color_shape_analysis_log = rt_results["analysis_log"]
color_shape_rt_planned_qc = rt_results["planned_rt_qc"]

print("Planned RT-cleaned trial table:", color_shape_rt_planned_cleaned.shape)
print("Reaction time summary:", color_shape_rt_summary.shape)
print("Analysis log:", color_shape_analysis_log.shape)
print("Saved figure:", rt_results["output_paths"]["planned_rt_qc_figure"])

## RT and Analysis Log Checks

Inspect trial removal percentages, final congruent/incongruent RT, and the combined analysis log.

In [ ]:
display(
    color_shape_rt_low_rt_summary
    .assign(percent_low_rt_removed=lambda df: 100 * df["pct_low_rt_trials_removed_of_correct"])
    .sort_values("percent_low_rt_removed", ascending=False)
)

display(
    color_shape_rt_mad_summary
    .assign(percent_mad_removed=lambda df: 100 * df["pct_mad_rt_trials_removed_of_pre_mad"])
    .sort_values("percent_mad_removed", ascending=False)
)

display(
    color_shape_rt_overall_removal_summary
    .assign(percent_total_removed=lambda df: 100 * df["pct_total_trials_removed"])
    .sort_values("percent_total_removed", ascending=False)
)

display(color_shape_rt_summary)

display(color_shape_analysis_log)

display(color_shape_rt_planned_qc.sort_values("mean_rt_ms"))

## Pre/Post Accuracy and RT Figure

Create the 2 × 2 pre/post figure for accuracy and RT by group and congruency, using subject-level bootstrap confidence intervals.

In [ ]:
color_shape_summary_long = color_shape.create_color_shape_summary_long(
    color_shape_analysis_log
)
color_shape_group_summary = color_shape.summarize_color_shape_group_level(
    color_shape_summary_long
)
fig, axes, color_shape_prepost_figure_path = color_shape.plot_color_shape_prepost_accuracy_rt_by_group(
    color_shape_summary_long,
    n_boot=5000,
)

display(color_shape_summary_long)
display(color_shape_group_summary)
print("Saved figure:", color_shape_prepost_figure_path)

## Mixed ANOVAs

Run mixed ANOVAs separately for congruent and incongruent accuracy and RT.

In [ ]:
color_shape_accuracy_anova = color_shape.run_color_shape_mixed_anova(
    color_shape_summary_long,
    dv="accuracy",
    label="accuracy",
)
color_shape_rt_anova = color_shape.run_color_shape_mixed_anova(
    color_shape_summary_long,
    dv="rt_correct",
    label="RT",
)

display(color_shape_accuracy_anova)
display(color_shape_rt_anova)